# MVP — Predição de Severidade de Ocorrências Aeronáuticas por Tipo de Operação

**Aluna:** Mariane Oliveira  
**Curso:** PUC — Sprint de Machine Learning  
**Dataset:** CENIPA/FAB — Centro de Investigação e Prevenção de Acidentes Aeronáuticos  
**Data:** Junho 2026

---
## Seção 1 — Definição do Problema

### Contexto

O Brasil possui um dos maiores espaços aéreos do mundo. O CENIPA/FAB registra todas as ocorrências aeronáuticas (acidentes e incidentes) desde os anos 2000. Compreender os fatores que determinam a severidade dessas ocorrências é fundamental para políticas de segurança e prevenção.

### Pergunta de Pesquisa

**Principal:** É possível prever a severidade de uma ocorrência aeronáutica com base nas características da aeronave e da operação?

**Secundária:** Quais tipos de operação apresentam maior risco (maior proporção de danos severos)?

### Abordagem Híbrida

O projeto é dividido em duas fases:

- **Fase 1 (Semana 2):** Prever o tipo de operação (`aeronave_tipo_operacao`) a partir das características da ocorrência.
- **Fase 2 (Semana 3):** Para cada tipo de operação, prever a severidade do dano (`aeronave_nivel_dano`).

### Variável-Alvo

- **Fase 1:** `aeronave_tipo_operacao` (REGULAR, TÁXI AÉREO, AGRÍCOLA, PRIVADA, etc.)
- **Fase 2:** `aeronave_nivel_dano` (NENHUM, LEVE, SUBSTANCIAL, DESTRUÍDA)

### Resultado Esperado

Um ranking de risco por tipo de operação, com recomendações regulatórias baseadas em dados.

---
## Seção 2 — Descrição dos Dados

### Fonte

Os dados são públicos e fornecidos pelo **CENIPA/FAB** (Centro de Investigação e Prevenção de Acidentes Aeronáuticos), disponíveis em: https://www.gov.br/transportes/pt-br/assuntos/transporte-aereo/seguranca-de-aviacao/ocorrencias-aeronauticas

### Tabelas Utilizadas

| Arquivo | Descrição | Linhas (~) |
|---|---|---|
| `ocorrencia.csv` | Registro geral das ocorrências (local, data, classificação) | 13.185 |
| `aeronave.csv` | Dados da aeronave envolvida (tipo, operação, fase, dano) | 13.301 |
| `ocorrencia_tipo.csv` | Tipo e categoria da ocorrência (taxonomia ICAO) | 13.900 |
| `fator_contribuinte.csv` | Fatores que contribuíram para a ocorrência | 8.613 |

### Chaves de Junção

- `ocorrencia.codigo_ocorrencia` → `aeronave.codigo_ocorrencia2`
- `ocorrencia.codigo_ocorrencia` → `ocorrencia_tipo.codigo_ocorrencia1`

### Variáveis de Interesse

| Variável | Tabela | Tipo | Papel |
|---|---|---|---|
| `aeronave_tipo_operacao` | aeronave | Categórica | Target Fase 1 |
| `aeronave_nivel_dano` | aeronave | Categórica | Target Fase 2 |
| `ocorrencia_classificacao` | ocorrencia | Categórica | Feature |
| `aeronave_fase_operacao` | aeronave | Categórica | Feature |
| `aeronave_motor_tipo` | aeronave | Categórica | Feature |
| `aeronave_motor_quantidade` | aeronave | Categórica | Feature |
| `aeronave_pmd_categoria` | aeronave | Numérica | Feature |
| `aeronave_fatalidades_total` | aeronave | Numérica | Feature |
| `ocorrencia_uf` | ocorrencia | Categórica | Feature |
| `ocorrencia_tipo_categoria` | ocorrencia_tipo | Categórica | Feature |

### 2.1 — Setup: Bibliotecas e Configurações

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

# Reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Estilo de visualização
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13

print('Bibliotecas carregadas com sucesso.')
print(f'NumPy: {np.__version__} | Pandas: {pd.__version__}')

### 2.2 — Carga dos Dados

In [ ]:
# Ajuste o caminho conforme necessário
DATA_PATH = 'raw/CENIPA_FAB/'

ocorrencia = pd.read_csv(
    DATA_PATH + 'ocorrencia.csv',
    sep=';',
    encoding='latin-1',
    low_memory=False
)

aeronave = pd.read_csv(
    DATA_PATH + 'aeronave.csv',
    sep=';',
    encoding='latin-1',
    low_memory=False
)

ocorrencia_tipo = pd.read_csv(
    DATA_PATH + 'ocorrencia_tipo.csv',
    sep=';',
    encoding='latin-1',
    low_memory=False
)

fator = pd.read_csv(
    DATA_PATH + 'fator_contribuinte.csv',
    sep=';',
    encoding='latin-1',
    low_memory=False
)

print(f'ocorrencia:      {ocorrencia.shape}')
print(f'aeronave:        {aeronave.shape}')
print(f'ocorrencia_tipo: {ocorrencia_tipo.shape}')
print(f'fator:           {fator.shape}')

In [ ]:
# Junção ocorrencia + aeronave
df = ocorrencia.merge(
    aeronave,
    left_on='codigo_ocorrencia',
    right_on='codigo_ocorrencia2',
    how='inner'
)

# Junção com ocorrencia_tipo (pode haver múltiplos tipos por ocorrência — pegamos o primeiro)
tipo_dedup = ocorrencia_tipo.drop_duplicates(subset='codigo_ocorrencia1', keep='first')
df = df.merge(
    tipo_dedup[['codigo_ocorrencia1', 'ocorrencia_tipo_categoria', 'taxonomia_tipo_icao']],
    left_on='codigo_ocorrencia',
    right_on='codigo_ocorrencia1',
    how='left'
)

print(f'Dataset final: {df.shape[0]} linhas, {df.shape[1]} colunas')
df.head(3)

---
## Seção 3 — Análise Exploratória dos Dados (EDA)

### 3.1 — Visão Geral do Dataset

In [ ]:
print('=== Tipos de dados ===')
print(df.dtypes.value_counts())
print()
print('=== Primeiras linhas ===')
df.head(3)

In [ ]:
# Estatísticas das colunas numéricas
df.describe(include='all').T[['count', 'unique', 'top', 'freq', 'mean', 'std', 'min', 'max']]

### 3.2 — Análise de Valores Ausentes

O CENIPA usa `***` como placeholder para dados não informados. Precisamos tratar isso como `NaN`.

In [ ]:
# Substituir '***' e '' por NaN
df.replace(['***', '', 'NULL'], np.nan, inplace=True)

# Percentual de valores ausentes por coluna
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_df = missing[missing > 0].reset_index()
missing_df.columns = ['coluna', 'pct_ausente']

plt.figure(figsize=(12, 6))
sns.barplot(
    data=missing_df.head(20),
    x='pct_ausente', y='coluna',
    palette='Reds_r'
)
plt.title('Top 20 Colunas com Valores Ausentes (%)')
plt.xlabel('% Ausente')
plt.tight_layout()
plt.show()

print(f'\nColunas com >50% ausentes: {(missing > 50).sum()}')
print(f'Colunas sem ausentes:      {(missing == 0).sum()}')

### 3.3 — Distribuição da Variável-Alvo (Fase 1): Tipo de Operação

In [ ]:
target_counts = df['aeronave_tipo_operacao'].value_counts(dropna=False)
print('Distribuição de aeronave_tipo_operacao:')
print(target_counts)
print(f'\nTotal: {target_counts.sum()} | Classes: {df["aeronave_tipo_operacao"].nunique(dropna=True)}')

In [ ]:
plot_data = df['aeronave_tipo_operacao'].value_counts().reset_index()
plot_data.columns = ['tipo_operacao', 'count']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot
sns.barplot(
    data=plot_data, x='count', y='tipo_operacao',
    ax=axes[0], palette='Blues_r'
)
axes[0].set_title('Ocorrências por Tipo de Operação')
axes[0].set_xlabel('Quantidade')
axes[0].set_ylabel('')

# Pizza
axes[1].pie(
    plot_data['count'],
    labels=plot_data['tipo_operacao'],
    autopct='%1.1f%%',
    startangle=90
)
axes[1].set_title('Proporção por Tipo de Operação')

plt.tight_layout()
plt.show()

print('\nObservação: Há desbalanceamento de classes — será necessário estratificação no split e uso de métricas ponderadas (F1-weighted).')

### 3.4 — Distribuição da Variável-Alvo (Fase 2): Nível de Dano

In [ ]:
dano_counts = df['aeronave_nivel_dano'].value_counts(dropna=False)
print('Distribuição de aeronave_nivel_dano:')
print(dano_counts)

plt.figure(figsize=(8, 4))
sns.barplot(
    x=dano_counts.index.tolist(),
    y=dano_counts.values,
    palette='OrRd'
)
plt.title('Distribuição do Nível de Dano (Target Fase 2)')
plt.xlabel('Nível de Dano')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.show()

### 3.5 — Severidade por Tipo de Operação

Esta é a análise central do projeto: qual tipo de operação tem maior proporção de danos graves?

In [ ]:
# Crosstab: tipo de operação x nível de dano
cross = pd.crosstab(
    df['aeronave_tipo_operacao'],
    df['aeronave_nivel_dano'],
    normalize='index'
) * 100

# Ordenar por coluna 'DESTRUÍDA' se existir
if 'DESTRUÍDA' in cross.columns:
    cross = cross.sort_values('DESTRUÍDA', ascending=False)

cross.plot(kind='bar', stacked=True, figsize=(12, 5), colormap='RdYlGn_r')
plt.title('Distribuição de Dano por Tipo de Operação (%)')
plt.ylabel('%')
plt.xlabel('Tipo de Operação')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Nível de Dano', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

print('\n% por tipo de operação:')
cross.round(1)

### 3.6 — Análise das Features Candidatas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Fase da operação
fase_counts = df['aeronave_fase_operacao'].value_counts().head(10)
sns.barplot(x=fase_counts.values, y=fase_counts.index, ax=axes[0], palette='Blues_r')
axes[0].set_title('Top 10 Fases de Operação')
axes[0].set_xlabel('Quantidade')

# Tipo de motor
motor_counts = df['aeronave_motor_tipo'].value_counts().head(8)
sns.barplot(x=motor_counts.values, y=motor_counts.index, ax=axes[1], palette='Greens_r')
axes[1].set_title('Tipo de Motor')
axes[1].set_xlabel('Quantidade')

# Classificação da ocorrência
class_counts = df['ocorrencia_classificacao'].value_counts()
sns.barplot(x=class_counts.values, y=class_counts.index, ax=axes[2], palette='Oranges_r')
axes[2].set_title('Classificação da Ocorrência')
axes[2].set_xlabel('Quantidade')

plt.tight_layout()
plt.show()

In [ ]:
# Fatalidades — variável numérica
df['aeronave_fatalidades_total'] = pd.to_numeric(
    df['aeronave_fatalidades_total'], errors='coerce'
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribuição de fatalidades
df['aeronave_fatalidades_total'].clip(upper=10).hist(
    bins=10, ax=axes[0], color='salmon', edgecolor='white'
)
axes[0].set_title('Distribuição de Fatalidades (clip=10)')
axes[0].set_xlabel('Fatalidades')
axes[0].set_ylabel('Frequência')

# Fatalidades médias por tipo de operação
fat_by_op = df.groupby('aeronave_tipo_operacao')['aeronave_fatalidades_total'].mean().sort_values(ascending=False)
fat_by_op.plot(kind='bar', ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title('Fatalidades Médias por Tipo de Operação')
axes[1].set_ylabel('Média de Fatalidades')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 3.7 — Análise Temporal

In [ ]:
# Extrair ano da data da ocorrência
df['ocorrencia_dia'] = pd.to_datetime(df['ocorrencia_dia'], format='%d/%m/%Y', errors='coerce')
df['ano'] = df['ocorrencia_dia'].dt.year

# Ocorrências por ano
por_ano = df.groupby('ano').size().reset_index(name='total')
por_ano = por_ano[por_ano['ano'].between(2010, 2025)]

plt.figure(figsize=(12, 4))
sns.lineplot(data=por_ano, x='ano', y='total', marker='o', color='steelblue')
plt.title('Ocorrências por Ano (2010–2025)')
plt.ylabel('Total de Ocorrências')
plt.xlabel('Ano')
plt.xticks(por_ano['ano'].astype(int))
plt.tight_layout()
plt.show()

---
## Seção 4 — Preparação dos Dados

### 4.1 — Seleção de Features e Targets

Baseado na EDA, selecionamos features com boa cobertura e relevância para o problema.

In [ ]:
FEATURES = [
    'ocorrencia_classificacao',   # ACIDENTE / INCIDENTE / INCIDENTE GRAVE
    'aeronave_fase_operacao',     # DECOLAGEM, CRUZEIRO, POUSO, etc.
    'aeronave_motor_tipo',        # PISTÃO, TURBOÉLICE, JATO, etc.
    'aeronave_motor_quantidade',  # MONOMOTOR, BIMOTOR, etc.
    'aeronave_nivel_dano',        # NENHUM, LEVE, SUBSTANCIAL, DESTRUÍDA
    'aeronave_fatalidades_total', # numérica
    'ocorrencia_tipo_categoria',  # categoria ICAO da ocorrência
    'ocorrencia_uf',              # estado da ocorrência
]

TARGET_FASE1 = 'aeronave_tipo_operacao'
TARGET_FASE2 = 'aeronave_nivel_dano'

# Dataset de trabalho
df_model = df[FEATURES + [TARGET_FASE1]].copy()

print(f'Shape antes de limpeza: {df_model.shape}')
print(f'\nNaN por coluna:')
print(df_model.isnull().sum().sort_values(ascending=False))

### 4.2 — Limpeza e Tratamento de Missing Values

In [ ]:
# Remover linhas onde o TARGET é nulo (não há o que prever)
df_model = df_model.dropna(subset=[TARGET_FASE1])

# Remover classes muito raras (menos de 30 exemplos) — inviáveis para treino
min_samples = 30
class_counts = df_model[TARGET_FASE1].value_counts()
classes_validas = class_counts[class_counts >= min_samples].index
df_model = df_model[df_model[TARGET_FASE1].isin(classes_validas)]

print(f'Shape após limpeza: {df_model.shape}')
print(f'\nClasses mantidas ({len(classes_validas)}):')
print(df_model[TARGET_FASE1].value_counts())

### 4.3 — Encoding de Variáveis Categóricas

Usamos `pd.get_dummies` (One-Hot Encoding) para variáveis categóricas nominais. A variável numérica (`aeronave_fatalidades_total`) recebe imputação pela mediana antes do split para não vazar informação — como é uma estatística da distribuição de treino, fazemos a imputação global aqui por simplicidade, e refazemos corretamente após o split na pipeline.

In [ ]:
# Converter fatalidades para numérico
df_model['aeronave_fatalidades_total'] = pd.to_numeric(
    df_model['aeronave_fatalidades_total'], errors='coerce'
)

# Separar X e y antes de qualquer transformação
FEAT_CAT = [
    'ocorrencia_classificacao',
    'aeronave_fase_operacao',
    'aeronave_motor_tipo',
    'aeronave_motor_quantidade',
    'aeronave_nivel_dano',
    'ocorrencia_tipo_categoria',
    'ocorrencia_uf',
]
FEAT_NUM = ['aeronave_fatalidades_total']

X = df_model[FEAT_CAT + FEAT_NUM].copy()
y = df_model[TARGET_FASE1].copy()

print(f'X: {X.shape} | y: {y.shape}')
print(f'Classes: {sorted(y.unique())}')

### 4.4 — Divisão Treino / Teste

**Importante:** O split é feito ANTES do pré-processamento para evitar data leakage. Usamos `stratify=y` para manter a proporção de classes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')
print(f'Proporção treino/teste: {X_train.shape[0]/len(X)*100:.0f}% / {X_test.shape[0]/len(X)*100:.0f}%')
print()
print('Distribuição de classes no treino:')
print(y_train.value_counts(normalize=True).round(3))

### 4.5 — Pipeline de Pré-processamento

O pipeline é ajustado (`fit`) apenas nos dados de treino e aplicado (`transform`) no teste. Isso previne data leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Transformador numérico: imputar mediana
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Transformador categórico: imputar moda + OHE
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, FEAT_NUM),
    ('cat', cat_transformer, FEAT_CAT),
])

# Ajustar APENAS no treino
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

print(f'X_train pré-processado: {X_train_prep.shape}')
print(f'X_test  pré-processado: {X_test_prep.shape}')
print('\n✅ Pipeline ajustado no treino e aplicado ao teste — sem data leakage.')

### 4.6 — Encoding do Target

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print('Mapeamento de classes:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

### 4.7 — Resumo da Preparação

| Etapa | Decisão | Justificativa |
|---|---|---|
| Missing values categóricos | Imputação pela moda | Preserva distribuição das classes |
| Missing values numéricos | Imputação pela mediana | Robusta a outliers |
| Encoding categórico | OneHotEncoding | Evita ordinalidade artificial |
| Classes raras (<30) | Removidas | Inviáveis para treino/avaliação |
| Split | 80/20 estratificado | Mantém proporção de classes |
| Ordem de operações | Split → Fit → Transform | Previne data leakage |

**Próximo passo (Seção 5):** Treinar os modelos de Fase 1 — Regressão Logística, Random Forest, XGBoost.

---
## Seção 5 — Modelagem (Fase 1): Previsão do Tipo de Operação

> 🔜 **A ser desenvolvida na Semana 2** — Modelos: Regressão Logística → Random Forest → XGBoost

---
## Seção 6 — Avaliação (Fase 1)

> 🔜 **A ser desenvolvida na Semana 2** — Métricas, matriz de confusão, importância de features

---
## Seção 7 — Otimização de Hiperparâmetros

> 🔜 **A ser desenvolvida na Semana 3** — GridSearchCV no melhor modelo

---
## Seção 8 — Modelagem (Fase 2): Severidade por Tipo de Operação

> 🔜 **A ser desenvolvida na Semana 3** — Modelos separados por tipo de operação

---
## Seção 9 — Avaliação (Fase 2)

> 🔜 **A ser desenvolvida na Semana 3** — Ranking de risco por tipo de operação

---
## Seção 10 — Conclusão

> 🔜 **A ser desenvolvida na Semana 4** — Achados, limitações, próximos passos